In [3]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ExpSineSquared, WhiteKernel
from gpr_predictions import Halogenation
import warnings

warnings.filterwarnings("ignore") # Ignore convergence warnings when running the GPR models

Data imports

In [44]:
data_yield = pd.read_csv('hte_halogenation_yields_for_gpr.csv', index_col = 'Unnamed: 0')
data_conv = pd.read_csv('hte_halogenation_conversions_for_gpr.csv', index_col = 'Unnamed: 0')

# Seperate the yield data into the different halogenations
#chloro_yield = data_yield[data_yield['Unnamed: 0')].str.contains('chloro')]
#bromo_yield = data_yield[data_yield['Unnamed: 0')].str.contains('bromo')]
#iodo_yield = data_yield[data_yield['Unnamed: 0')].str.contains('iodo')]

# Seperate the conversion data into the different halogenations
#chloro_conv = data_conv[data_conv['Unnamed: 0')].str.contains('chloro')]
#bromo_conv = data_conv[data_conv['Unnamed: 0')].str.contains('bromo')]
#iodo_conv = data_conv[data_conv['Unnamed: 0')].str.contains('iodo')]

# GPR Predictions for Stage 2 halogenation conditions prediction
This notebook contains original code use to determine optimum acidic conditions for the late stage halogenation of pharmaceuticals. This uses stage 2 data generated from screening a series of halogenations against 0 - 25 TFA equivalents, outputting information about the optimum conditions determined using an optimal kernel, as well as yield and conversion predicitions, at two different length scales.

Function for kernel screening using the GPR halogenation class

In [45]:
def gpr_kernel_screening(data_yield, 
                         data_conv, 
                         kernels,   
                         ):
    
    # Initialise dataframe to store condition prediction information
    optimum_condition_df = pd.DataFrame(columns=['substrate',
                                                 'reagent',
                                                 'pred_opt_tfa', 
                                                'pred_opt_yield(%)',
                                                'pred_opt_conv(%)',
                                                'kernel_yield',
                                                'kernel_yield_loo_mae(%)',
                                                'kernel_conv',
                                                'kernel_conv_loo_mae(%)',
                                            ]) 
    
    # Initialise dataframe to store the optimum conditions and kernel information
    gpr_pred_df_yield = pd.DataFrame(columns=np.linspace(0, 25, 1000))
    gpr_pred_df_conv = pd.DataFrame(columns=np.linspace(0, 25, 1000))


    acid_exp = [float(x) for x in data_yield.columns]
    #print(acid_exp)

    # Loop through the different halogenations
    for product_y, product_c in zip(data_yield.itertuples(index = True), data_conv.itertuples(index = True)):
        
        # Get the product name and extract the reagent and starting material
        product = product_y[0]
        starting_material = product.split('-')[-1]
        suffix = product.split('-')[-2]
        if suffix == 'chloro':
            reagent = "Palau'chlor"
        elif suffix == 'bromo':
            reagent = "N-bromosuccinimide"
        elif suffix == 'iodo':
            reagent = "N-iodosuccinimide"
        else:
            reagent = 'Unknown'
        
        print(f'Predicting conditions for the formation of {product_y[0]} with {starting_material} and {reagent}...')

        # Get the yield and conversion data
        yield_data = product_y[1:]
        #print(f'Yield data for {product}: {yield_data}')
        conv_data = product_c[1:]
        
        combined_data = np.column_stack([acid_exp, yield_data, conv_data]) # Combine the yield and conversion data with the acid equiv data
        #print(combined_data)
        #print(f'Predicting conditions for the fomration of {product}...')

        # Initialise the Halogenation class
        halogenation = Halogenation(starting_material, reagent, combined_data)
        
        # Kernel screening
        for kernel_label, kernel in kernels.items():
            halogenation.gprcalculate(kernel, dset = 'both', kernelname = kernel_label) # Run all kernels specified prior
        
        # Get the best kernels (lowest MAE) for yield and conversion
        best_kernel_yield = halogenation.best_model_yield()
        best_kernel_conv = halogenation.best_model_conv()

        # Determine the optimum conditions for the best kernel
        optimum_condition = halogenation.pickoptimum(yield_kernel = best_kernel_yield, conv_kernel = best_kernel_conv) # Get the optimum conditions for the best kernel

        # Store the optimum conditions and kernel information
        optimum_condition_df.loc[product] = [starting_material,
                                             reagent,
                                             optimum_condition[0], 
                                             optimum_condition[1],
                                             optimum_condition[2],
                                             best_kernel_yield,
                                             halogenation.yieldoutputs[best_kernel_yield]['mae'],
                                             best_kernel_conv,
                                             halogenation.convoutputs[best_kernel_conv]['mae'],
                                          ]


        # Store the best kernel predicitions for plotting
        best_predictions_yield = halogenation.yieldoutputs[best_kernel_yield]['prediction'][:,1]
        best_predicitions_conv = halogenation.convoutputs[best_kernel_conv]['prediction'][:,1]
        gpr_pred_df_yield.loc[product] = best_predictions_yield # Append to yield df specified as an argument
        gpr_pred_df_conv.loc[product] = best_predicitions_conv # Append to conversion df specified as an argument


    return optimum_condition_df, gpr_pred_df_yield, gpr_pred_df_conv




Function to generate kernels for screening

In [46]:
# Kernels for screening - RBF, Matern (nu = 3/2), Matern (nu = 5/2) with WhiteKernel to model noise
def get_kernels(ls_bounds):
    return {'RBF': 1.0 * RBF(length_scale=1e0, length_scale_bounds=ls_bounds)+ WhiteKernel(noise_level=1e0, noise_level_bounds=ls_bounds),
          'Matern (nu = 3/2)': 1.0 * Matern(length_scale=1e0, length_scale_bounds=ls_bounds, nu=3/2)+ WhiteKernel(noise_level=1e0, noise_level_bounds=ls_bounds),
          'Matern (nu = 5/2)': 1.0 * Matern(length_scale=1e0, length_scale_bounds=ls_bounds, nu=5/2)+ WhiteKernel(noise_level=1e0, noise_level_bounds=ls_bounds),
          # 'Rational quadratic (RQ)': 1.0 * RationalQuadratic(length_scale=1e0, length_scale_bounds=ls_bounds),+ WhiteKernel(noise_level=1e0, noise_level_bounds=ls_bounds),
          # UNCOMMENT THE ABOVE LINES TO USE OTHER KERNELS. Tended to result in overfitting for many of the datasets used in this study
          }



Running the models and getting our predictions

In [47]:
#Screen kernels over 2 length scales. Compare the best from both length scales an choose based on inspection of the predictions
low_ls_kernels = get_kernels((1e-1, 1e1)) # Sometime a tendency to overfit with low length scales
high_ls_kernels = get_kernels((1e0, 1e2)) # Sometimes a tendency to underfit with high length scales

# Run the models!
optimum_condition_df_lowls, gpr_pred_df_yield_lowls, gpr_pred_df_conv_lowls = gpr_kernel_screening(data_yield, data_conv, low_ls_kernels)
optimum_condition_df_highls, gpr_pred_df_yield_highls, gpr_pred_df_conv_highls = gpr_kernel_screening(data_yield, data_conv, high_ls_kernels)

Predicting conditions for the formation of bromo-Bifonazole with Bifonazole and N-bromosuccinimide...
Predicting conditions for the formation of bromo-Camptothecin with Camptothecin and N-bromosuccinimide...
Predicting conditions for the formation of bromo-Ciprofloxacin with Ciprofloxacin and N-bromosuccinimide...
Predicting conditions for the formation of 4-bromo-Indomethacin with Indomethacin and N-bromosuccinimide...
Predicting conditions for the formation of 6-bromo-Indomethacin with Indomethacin and N-bromosuccinimide...
Predicting conditions for the formation of bromo-Ketanserin with Ketanserin and N-bromosuccinimide...
Predicting conditions for the formation of bromo-Lidocaine with Lidocaine and N-bromosuccinimide...
Predicting conditions for the formation of bromo-Mizolastine with Mizolastine and N-bromosuccinimide...
Predicting conditions for the formation of bromo-Nevirapine with Nevirapine and N-bromosuccinimide...
Predicting conditions for the formation of bromo-Palbociclib

Exporting the results for later use...

In [19]:
# Export the results to csvs
os.makedirs('gpr_results', exist_ok=True)
optimum_condition_df_lowls.to_csv('gpr_results/optimum_conditions_lowls.csv') # Optimum conditions using the lower length scale kernels (0.1 - 10)
gpr_pred_df_yield_lowls.to_csv('gpr_results/gpr_predictions_yield_lowls.csv') # GPR predictions for yield using the lower length scale kernels (0.1 - 10)
gpr_pred_df_conv_lowls.to_csv('gpr_results/gpr_predictions_conv_lowls.csv') # GPR predictions for conversion using the lower length scale kernels (0.1 - 10)

optimum_condition_df_highls.to_csv('gpr_results/optimum_conditions_highls.csv') # Optimum conditions using the higher length scale kernels (1 - 100)
gpr_pred_df_yield_highls.to_csv('gpr_results/gpr_predictions_yield_highls.csv') # GPR predictions for yield using the higher length scale kernels (1 - 100)
gpr_pred_df_conv_highls.to_csv('gpr_results/gpr_predictions_conv_highls.csv') # GPR predictions for conversion using the higher length scale kernels (1 - 100)